In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  WEEK-2: PROVIDER-AGNOSTIC LLM CLIENT — FINAL VERSION
# ══════════════════════════════════════════════════════════════════════
import os, time, subprocess, textwrap, requests, sys

# ─── 1. Project folder ────────────────────────────────────────────────
os.makedirs("/content/week-2", exist_ok=True)
os.chdir("/content/week-2")
print("📁 Working directory:", os.getcwd())

# ─── 2. Install Ollama ────────────────────────────────────────────────
print("\n⚙️  [1/7] Installing Ollama ...")
subprocess.run("apt-get -qq install -y zstd", shell=True, capture_output=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
               shell=True, capture_output=True)
print("   ", subprocess.run("ollama --version", shell=True,
                             capture_output=True, text=True).stdout.strip())

# ─── 3. Start the Ollama server in the background ─────────────────────
print("\n🚀 [2/7] Starting Ollama server ...")
subprocess.Popen("ollama serve", shell=True,
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

ready = False
for i in range(12):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=3)
        if r.ok:
            print(f"   ✅ Server ready after ~{i*5}s")
            ready = True
            break
    except (requests.exceptions.ConnectionError, requests.exceptions.Timeout):
        pass
    time.sleep(5)

if not ready:
    raise SystemExit("⚠️ Aborting: Ollama server not reachable.")

# ─── 4. Download Llama 3.2 (~2 GB) ────────────────────────────────────
print("\n📥 [3/7] Pulling llama3.2 (slow part, ~2 GB) ...")
subprocess.run("ollama pull llama3.2", shell=True)

# ─── 5. Create .env with the Gemini key ───────────────────────────────
print("\n🔑 [4/7] Creating .env ...")
# ▼ YOUR NEW KEY IS HARDCODED HERE
FALLBACK_KEY = "AQ.Ab8RN6IwVLOibH-w92q2IscpRoqf552AfOnc3kb4zhk6PF0liQ"

key = None
try:
    from google.colab import userdata
    key = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

if not key:
    key = FALLBACK_KEY

with open(".env", "w") as f:
    f.write(f'GEMINI_API_KEY="{key}"\n')

# ─── 6. requirements.txt + install ────────────────────────────────────
print("\n📦 [5/7] Installing Python dependencies ...")
with open("requirements.txt", "w") as f:
    f.write("python-dotenv\npytest\nrequests\n")
subprocess.run("pip install -q -r requirements.txt", shell=True)
# Force upgrade the google-genai SDK for retry logic
subprocess.run("pip install -q -U google-genai", shell=True)

# ─── 7. Write llm_client.py / main.py / test_client.py ───────────────
print("\n📝 [6/7] Writing project files ...")

with open("llm_client.py", "w") as f:
    f.write(textwrap.dedent('''
        from abc import ABC, abstractmethod
        from google import genai
        from google.genai import types
        import requests

        class LLMProvider(ABC):
            @abstractmethod
            def generate(self, prompt):
                pass

        class GeminiProvider(LLMProvider):
            def __init__(self, api_key, model="gemini-3.8-flash"):
                # Enable automatic exponential backoff for 503/429 errors
                retry_config = types.HttpRetryOptions(
                    attempts=5,
                    exp_base=2,
                    initial_delay=1,
                    http_status_codes=[429, 500, 503, 504]
                )
                self.client = genai.Client(
                    api_key=api_key,
                    http_options=types.HttpOptions(retry_options=retry_config)
                )
                self.model = model

            def generate(self, prompt):
                response = self.client.models.generate_content(
                    model=self.model,
                    contents=prompt,
                )
                return response.text

        class OllamaProvider(LLMProvider):
            def __init__(self, model="llama3.2"):
                self.model = model
                self.url = "http://localhost:11434/api/generate"

            def generate(self, prompt):
                response = requests.post(
                    self.url,
                    json={"model": self.model, "prompt": prompt, "stream": False},
                    timeout=120,
                )
                response.raise_for_status()
                return response.json()["response"]

        class LLMClient:
            def __init__(self, provider):
                self.provider = provider

            def generate(self, prompt):
                return self.provider.generate(prompt)
    ''').lstrip())

with open("main.py", "w") as f:
    f.write(textwrap.dedent('''
        import os
        from dotenv import load_dotenv
        from llm_client import GeminiProvider, OllamaProvider, LLMClient

        load_dotenv(override=True)

        provider_name = input("Choose provider (gemini/ollama): ").strip().lower()

        if provider_name == "gemini":
            api_key = os.getenv("GEMINI_API_KEY")
            if not api_key:
                raise ValueError("Gemini API key not found in .env")
            provider = GeminiProvider(api_key)
        elif provider_name == "ollama":
            provider = OllamaProvider("llama3.2")
        else:
            raise ValueError("Invalid provider selected")

        client = LLMClient(provider)

        print("\\n✅ Setup complete. Type 'exit' or 'quit' to stop the program.")

        while True:
            prompt = input("\\nEnter your question: ")

            if prompt.strip().lower() in ['exit', 'quit']:
                print("Goodbye!")
                break

            if not prompt.strip():
                continue

            answer = client.generate(prompt)
            print("\\nLLM Response:")
            print(answer)
    ''').lstrip())

with open("test_client.py", "w") as f:
    f.write(textwrap.dedent('''
        import unittest
        from llm_client import LLMClient

        class FakeProvider:
            def generate(self, prompt):
                return "AI stands for Artificial Intelligence."

        class TestLLMClient(unittest.TestCase):
            def test_generate(self):
                client = LLMClient(FakeProvider())
                result = client.generate("Explain AI")
                self.assertEqual(result, "AI stands for Artificial Intelligence.")

        if __name__ == "__main__":
            unittest.main()
    ''').lstrip())

# ─── 8. Run the unit test ─────────────────────────────────────────────
print("\n🧪 [7/7] Running unit test ...")
subprocess.run("python -m pytest test_client.py -v", shell=True)

# ─── 9. Interactive run ───────────────────────────────────────────────
print("\n" + "=" * 60)
print(" INTERACTIVE RUN — answer the prompts below this cell")
print("=" * 60)

# Clear the old cached module from memory before running
if 'llm_client' in sys.modules:
    del sys.modules['llm_client']

exec(open("main.py").read(), {"__name__": "__main__", "__file__": "main.py"})

📁 Working directory: /content/week-2

⚙️  [1/7] Installing Ollama ...

🚀 [2/7] Starting Ollama server ...
   ✅ Server ready after ~5s

📥 [3/7] Pulling llama3.2 (slow part, ~2 GB) ...

🔑 [4/7] Creating .env ...

📦 [5/7] Installing Python dependencies ...

📝 [6/7] Writing project files ...

🧪 [7/7] Running unit test ...

 INTERACTIVE RUN — answer the prompts below this cell
Choose provider (gemini/ollama): ollama

✅ Setup complete. Type 'exit' or 'quit' to stop the program.

Enter your question: what is api


ReadTimeout: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=120)